# 자료 식별 검증 (v2.3.5-P0-ID)
합성 테스트 자료입니다. 실제 API를 호출하지 않으며 인증키가 필요 없습니다. 프로젝트 폴더에서 실행하세요.

## 1. 검증용 자료 준비

In [1]:
import pandas as pd
from catalog_identity import identity_rows, identity_table, filter_identities, CONTROL, TITLE, ROWKEY, LINK
from nanet_metadata import normalize_frame, THESIS, ARTICLE, BOOK, UNKNOWN
from nanet_search_flow import partition_catalog_records

# 실제 검색 결과가 아닌 테스트 목록입니다.
data = pd.DataFrame([
    {"CONTROL_NO": "KINX2026064882", "title": "예시 학술기사", "author": "예시저자", "year": 2026},
    {"CONTROL_NO": "KDMT1202600031734", "title": "예시 학위논문", "author": "예시저자", "year": 2026},
    {"CONTROL_NO": "UNKN2026000001", "title": "", "publisher": "예시기관", "year": 2025},
])
normalized = normalize_frame(data)
print(normalized[["controlno", "자료분류"]])

           controlno      자료분류
0     KINX2026064882      학술기사
1  KDMT1202600031734      학위논문
2     UNKN2026000001  자료유형 미확인


In [2]:
# 별도 검증 셀
assert len(normalized) == 3
assert normalized.iloc[0]["자료분류"] == ARTICLE
assert normalized.iloc[1]["자료분류"] == THESIS
assert normalized.iloc[2]["자료분류"] == UNKNOWN
print("자동분류 검증 통과")

자동분류 검증 통과


## 2. 표에 관리번호·제목 표시

In [3]:
table = identity_table(normalized)
display(table)

,자료 관리번호,자료명/저자사항,저자,발행연도,발행처·대학,자료유형,분류 근거·미확인 이유,공식 상세정보,정식 참고문헌,분류출처,API순번
0,KINX2026064882,예시 학술기사,예시저자,2026,발행처 미확인,학술기사,검증 사례 기반 접두어 KINX (세부 학위·심사 여부는 별도),https://dl.nanet.go.kr/search/searchInnerDetai...,"예시저자, “예시 학술기사”, [학술지명 미확인], [권 미확인], [호 미확인],...",관리번호,
1,KDMT1202600031734,예시 학위논문,예시저자,2026,발행처 미확인,학위논문,검증 사례 기반 접두어 KDMT (세부 학위·심사 여부는 별도),https://dl.nanet.go.kr/search/searchInnerDetai...,"예시저자, “예시 학위논문”, [학위종류 미확인], [학위수여 대학·대학원 미확인]...",관리번호,
2,UNKN2026000001,제목 미확인,저자 미확인,2025,예시기관,,공식 유형·검증된 관리번호·서지정보가 부족함,https://dl.nanet.go.kr/search/searchInnerDetai...,"[자료유형 미확인] [저자 미확인], “[제목 미확인]”, 2025년 (월 미확인)...",자료부족,


In [4]:
assert table.columns[0] == CONTROL
assert table.iloc[0][CONTROL] == "KINX2026064882"
assert table.iloc[2][TITLE] == "제목 미확인"
assert "정식 참고문헌" in table.columns
print("표 식별 정보 검증 통과")

표 식별 정보 검증 통과


## 3. 선택창의 표시 문구

In [5]:
items = identity_rows(normalized)
for item in items:
    print(item["label"])

KINX2026064882 | 예시 학술기사 | 예시저자 | 2026
KDMT1202600031734 | 예시 학위논문 | 예시저자 | 2026
UNKN2026000001 | 제목 미확인 | 저자 미확인 | 2025


In [6]:
assert items[0]["label"] == "KINX2026064882 | 예시 학술기사 | 예시저자 | 2026"
assert items[0][LINK].endswith("controlNo=KINX2026064882")
assert items[0][ROWKEY] != items[1][ROWKEY]
print("선택창 검증 통과")

선택창 검증 통과


## 4. 목록에서 관리번호로 찾기

In [7]:
matched = filter_identities(items, "UNKN2026000001")
print(matched[0]["label"])

UNKN2026000001 | 제목 미확인 | 저자 미확인 | 2025


In [8]:
assert len(matched) == 1
assert matched[0][CONTROL] == "UNKN2026000001"
assert len(normalized) == 3  # 필터만 적용하고 원본은 유지
print("목록 찾기 검증 통과")

목록 찾기 검증 통과


## 5. 같은 제목·관리번호 누락 검증

In [9]:
duplicates = identity_rows(pd.DataFrame([
    {"title": "같은 제목"},
    {"title": "같은 제목"},
]))
for item in duplicates:
    print(item["label"])

관리번호 미확인 | 같은 제목 | 저자 미확인 | 연도 미확인 | 목록 식별 69b230468c
관리번호 미확인 | 같은 제목 | 저자 미확인 | 연도 미확인 | 목록 식별 b230468c#2


In [10]:
assert len(duplicates) == 2
assert duplicates[0][ROWKEY] != duplicates[1][ROWKEY]
assert all(not item[LINK] for item in duplicates)
assert all(item[CONTROL] == "관리번호 미확인" for item in duplicates)
print("누락 식별자 검증 통과")

누락 식별자 검증 통과
